# Urban Sentinel — façade risk analysis

**Question.** Using only records available *before* a decision date, can we rank New York City buildings by the chance that their next façade inspection (FISP / Local Law 11) is filed **UNSAFE**?

**Design.** Train on inspection cycles 7–8 (2010–2019) and test on cycle 9 (2020–2024), whose outcomes the model never sees. Dated records are restricted to before each cycle's start. PLUTO attributes and building-to-lot links are a current snapshot, not a reconstructed historical snapshot.

**Data.** NYC Open Data: DOB NOW façade filings (`xubg-57si`), DOB and ECB violations, MapPLUTO building attributes. Models: XGBoost, scikit-learn; deployment inference on IBM LinuxONE (NNPA via IBM Snap ML).

In [ ]:
import platform, re, sys, time
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import sklearn
import xgboost as xgb
from sklearn.calibration import calibration_curve
from sklearn.metrics import average_precision_score, brier_score_loss, precision_recall_curve, roc_auc_score

REPO = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "ml" / "urban_sentinel").is_dir())
sys.path.insert(0, str(REPO / "ml"))
from urban_sentinel.config import CYCLE_START
from urban_sentinel.evaluation import precision_recall_at_k
from urban_sentinel.experiments import fit_all, make_dataset, score_methods
from urban_sentinel.featureset import Context
from urban_sentinel.ingest import load_dob_facade_risk
from urban_sentinel.labels import build_labels
from urban_sentinel.score import _describe

%config InlineBackend.figure_format = "retina"
plt.style.use("seaborn-v0_8-whitegrid")
plt.rcParams.update({
    "figure.dpi": 120, "figure.figsize": (8.5, 4.6), "font.size": 11,
    "axes.titlesize": 13, "axes.titleweight": "bold", "axes.titlelocation": "left",
    "axes.spines.top": False, "axes.spines.right": False, "legend.frameon": False,
})
pd.set_option("display.precision", 3)

MODEL, LOGIT, RULE = "Urban Sentinel (XGBoost)", "Logistic regression", "Last-status rule"
COLORS = {MODEL: "#0f62fe", LOGIT: "#8a3ffc", RULE: "#8d8d8d"}

cpuinfo = Path("/proc/cpuinfo").read_text() if Path("/proc/cpuinfo").exists() else ""
pd.DataFrame({"value": {
    "machine": platform.machine(),
    "byte order": sys.byteorder,
    "NNPA available (CPU flag)": bool(re.search(r"\bnnpa\b", cpuinfo)),
    "python": platform.python_version(),
    "pandas": pd.__version__, "scikit-learn": sklearn.__version__, "xgboost": xgb.__version__,
}})

## 1 · Data

In [ ]:
ctx = Context.load()          # cached NYC Open Data pulls (data/raw)
fac = ctx.fac

pd.DataFrame({"records": {
    "façade filings": len(fac),
    "buildings (BIN)": fac["bin"].nunique(),
    "DOB violations (façade-related)": len(ctx.events.get("dob", [])),
    "ECB violations (wall maintenance)": len(ctx.events.get("ecb", [])),
}})

In [ ]:
inspections = fac[fac["filing_type"].eq("Initial") & fac["filing_status"].isin(["SAFE", "SWARMP", "UNSAFE"])]
inspections[["bin", "cycle", "submitted_on", "filing_status", "exterior_wall_material_s_"]] \
    .rename(columns={"exterior_wall_material_s_": "wall material"}).sample(6, random_state=7)

In [ ]:
by_cycle = pd.DataFrame({
    c: {"buildings filed": len(lab), "UNSAFE share": lab["y"].mean()}
    for c in range(6, 11) if len(lab := build_labels(fac, c))
}).T.rename_axis("cycle")
by_cycle["buildings filed"] = by_cycle["buildings filed"].astype(int)

fig, ax = plt.subplots(figsize=(8.5, 3.6))
bars = ax.bar(by_cycle.index.astype(str), by_cycle["UNSAFE share"], color="#a8a8a8")
bars[-1].set_alpha(0.45)
ax.bar_label(bars, labels=[f"{v:.0%}" for v in by_cycle["UNSAFE share"]], padding=3)
ax.yaxis.set_major_formatter("{x:.0%}")
ax.set(title="Share of buildings whose façade inspection was filed UNSAFE",
       xlabel="FISP cycle  (cycle 10 still in progress)", ylabel="UNSAFE share", ylim=(0, 0.35))
ax.grid(axis="x", visible=False)
plt.show()
by_cycle.style.format({"UNSAFE share": "{:.1%}", "buildings filed": "{:,}"})

## 2 · Point-in-time features

Dated filing and violation features use records **before that cycle's start** (2010-02-21, 2015-02-21, 2020-02-21); cutoff checks apply to those records. PLUTO attributes and building-to-lot links use a current snapshot, a limitation of this retrospective test. The label is the worst Initial-filing status in the cycle (UNSAFE = 1).

In [ ]:
datasets = {c: make_dataset(ctx, c) for c in (7, 8, 9)}

pd.DataFrame({
    f"cycle {c}": {"decision date": CYCLE_START[c].date(), "buildings": len(lab),
                   "features": X.shape[1], "UNSAFE rate": f"{lab['y'].mean():.1%}"}
    for c, (X, lab) in datasets.items()
}).T

## 3 · Train on cycles 7–8

In [ ]:
(X7, l7), (X8, l8), (X9, l9) = datasets.values()

t0 = time.perf_counter()
model, calibrator, logit, rounds = fit_all([(X7, l7["y"]), (X8, l8["y"])], (X8, l8["y"]))
print(f"XGBoost: {rounds} boosting rounds (early stopping on cycle 8), "
      f"trained in {time.perf_counter() - t0:.1f} s on {platform.machine()}")
params = model.get_params()
pd.DataFrame({"value": {k: params[k] for k in ("n_estimators", "max_depth", "learning_rate", "subsample",
                                               "colsample_bytree", "min_child_weight", "reg_lambda")}})

## 4 · Blind test on cycle 9 (2020–2024)

In [ ]:
y = l9["y"].to_numpy()
raw = score_methods(X9, model, calibrator, logit)
scores = {MODEL: raw["xgboost"], LOGIT: raw["logistic_history"], RULE: raw["rule_prev_status"]}
prob = raw["xgboost_calibrated"]

def at_k(s, k, mask=slice(None)):
    return precision_recall_at_k(y[mask], s[mask], k)[0]

results = pd.DataFrame({
    name: {"PR-AUC": average_precision_score(y, s), "ROC-AUC": roc_auc_score(y, s),
           **{f"Precision @ {k}": at_k(s, k) for k in (250, 500, 1000)}}
    for name, s in scores.items()
}).T
results.loc["Random order"] = [y.mean(), 0.5, y.mean(), y.mean(), y.mean()]

(results.style
    .format("{:.3f}", subset=["PR-AUC", "ROC-AUC"])
    .format("{:.1%}", subset=[c for c in results if c.startswith("Precision")])
    .highlight_max(color="#d0e2ff"))

In [ ]:
fig, ax = plt.subplots()
for name, s in scores.items():
    p, r, _ = precision_recall_curve(y, s)
    ax.plot(r, p, color=COLORS[name], lw=2.2 if name == MODEL else 1.6,
            label=f"{name}   AP = {average_precision_score(y, s):.3f}")
ax.axhline(y.mean(), color="black", ls=":", lw=1.2, label=f"Random order   AP = {y.mean():.3f}")
ax.set(title="Precision–recall on the cycle-9 hold-out", xlabel="Recall  (share of all UNSAFE façades found)",
       ylabel="Precision", xlim=(0, 1), ylim=(0, 1))
ax.legend(loc="upper right")
plt.show()

In [ ]:
budget = np.arange(1, 2001)
fig, ax = plt.subplots()
for name, s in scores.items():
    order = np.lexsort((np.arange(len(s)), -s))
    found = np.cumsum(y[order])[: len(budget)]
    ax.plot(budget, found, color=COLORS[name], lw=2.2 if name == MODEL else 1.6,
            label=f"{name}  ·  {found[499]:.0f} in first 500")
ax.plot(budget, budget * y.mean(), color="black", ls=":", lw=1.2,
        label=f"Random order  ·  {500 * y.mean():.0f} in first 500")
ax.axvline(500, color="#c6c6c6", lw=1)
ax.set(title="UNSAFE façades found for a given inspection budget",
       xlabel="Buildings inspected, highest risk first", ylabel="UNSAFE façades found")
ax.legend(loc="upper left")
plt.show()

### Calibration

Probabilities are Platt-calibrated on cycle 8, so a predicted 30% should mean roughly 30% of such buildings file UNSAFE.

In [ ]:
observed, predicted = calibration_curve(y, prob, n_bins=10, strategy="quantile")
fig, ax = plt.subplots(figsize=(5.2, 5))
ax.plot([0, 0.7], [0, 0.7], color="#c6c6c6", ls="--", lw=1, label="Perfect calibration")
ax.plot(predicted, observed, "o-", color=COLORS[MODEL], lw=2, label="Urban Sentinel (deciles)")
ax.set(title="Predicted vs observed UNSAFE rate", xlabel="Mean predicted probability",
       ylabel="Observed UNSAFE rate", xlim=(0, 0.7), ylim=(0, 0.7), aspect="equal")
ax.legend(loc="upper left")
plt.show()
print(f"Brier score: {brier_score_loss(y, prob):.4f}   (always predicting the base rate: {brier_score_loss(y, np.full_like(prob, y.mean())):.4f})")

### Buildings that passed last time

The hardest and most useful case: buildings whose last inspection was **SAFE**. A last-status rule cannot rank them at all.

In [ ]:
prev_safe = ((X9["last_status"] == 0) & (X9["status_at_cutoff"] <= 0)).to_numpy()
subgroup = pd.DataFrame({
    name: {"Precision @ 250": at_k(s, 250, prev_safe), "PR-AUC": average_precision_score(y[prev_safe], s[prev_safe])}
    for name, s in scores.items()
}).T
subgroup.loc["Random order"] = [y[prev_safe].mean()] * 2
print(f"{prev_safe.sum():,} previously-SAFE buildings; {y[prev_safe].mean():.1%} filed UNSAFE in cycle 9")
subgroup.style.format("{:.1%}", subset=["Precision @ 250"]).format("{:.3f}", subset=["PR-AUC"]).highlight_max(color="#d0e2ff")

### Against NYC DOB's published risk score

DOB published façade risk probabilities in 2022 ([NYCDOB/FacadeRisk](https://github.com/NYCDOB/FacadeRisk)). Compared on the buildings both systems score. DOB's model may have seen some cycle-9 outcomes, so this comparison **favours DOB**.

In [ ]:
dob = load_dob_facade_risk().reindex(X9.index).to_numpy()
common = ~np.isnan(dob)
head_to_head = pd.DataFrame({
    name: {"Precision @ 500": at_k(s, 500, common), "PR-AUC": average_precision_score(y[common], s[common])}
    for name, s in {MODEL: scores[MODEL], "NYC DOB FacadeRisk (2022)": dob}.items()
}).T
head_to_head.loc["Random order"] = [y[common].mean()] * 2
print(f"{common.sum():,} buildings scored by both")
head_to_head.style.format("{:.1%}", subset=["Precision @ 500"]).format("{:.3f}", subset=["PR-AUC"]).highlight_max(color="#d0e2ff")

## 5 · What drives a score

In [ ]:
def label(feature):
    try:
        return _describe(feature, 0.0)[0]
    except Exception:
        return feature.replace("_", " ")

shap = model.get_booster().predict(xgb.DMatrix(X9), pred_contribs=True)[:, :-1]   # XGBoost TreeSHAP
importance = pd.Series(np.abs(shap).mean(axis=0), index=X9.columns).nlargest(12).iloc[::-1]

fig, ax = plt.subplots(figsize=(8.5, 5))
ax.barh([label(f) for f in importance.index], importance.to_numpy(), color=COLORS[MODEL])
ax.set(title="Top drivers of the cycle-9 score", xlabel="Mean |SHAP value|  (log-odds)")
ax.grid(axis="y", visible=False)
plt.show()

## 6 · Inference in this notebook's kernel

This optional section loads existing deployed artifacts from the configured data directory; it does not call the live API or use the evaluation model trained above. On a compatible IBM Z kernel, **IBM Snap ML on NNPA** (`zdnn_tensors`) is enabled only after a numerical parity check against XGBoost. Otherwise, only CPU timing is shown. These are model-only timings for historical cycle-9 feature rows, not the saved production benchmark or full API latency.

In [ ]:
from urban_sentinel.artifacts import DeployedModel, MODEL_FILE, CALIBRATION_FILE, META_FILE

import platform
from datetime import datetime, timezone
print(f'Current kernel: {platform.system()} / {platform.machine()} · {datetime.now(timezone.utc).isoformat()}')
if not all(path.is_file() for path in (MODEL_FILE, CALIBRATION_FILE, META_FILE)):
    raise RuntimeError('No complete deployed model artifacts found. Run the production training pipeline or set URBAN_SENTINEL_DATA_DIR to an existing complete run before starting this kernel. See ml/notebooks/README.md. This is not a successful live inference run.')
deployed = DeployedModel.load()
print(f'Loaded model: {deployed.meta["model_sha256"]}')
status = deployed.enable_nnpa(X9.iloc[:500])
sample = X9.iloc[:200]

def median_ms(engine):
    return np.median([deployed.score_live(sample.iloc[[i]], engine)["score_ms"] for i in range(len(sample))])

timing = {"XGBoost on CPU": median_ms("xgboost")}
if status["engine"] == "snapml-nnpa":
    timing["Snap ML on NNPA"] = median_ms("nnpa")
    print(f"NNPA enabled · zDNN {status['zdnn_version']} · tree format {status['tree_format']} · "
          f"max |Δ| vs XGBoost on 500 buildings: {status['parity_max_abs_diff']:.1e}")
else:
    print(f"NNPA not used on this machine: {status.get('reason')}")

pd.DataFrame({"median ms per building": timing}).round(3)

## Summary

* On outcomes it never saw, the model puts UNSAFE façades at the top of the inspection list far more often than the current last-status rule or random order (tables in §4).
* It ranks buildings that **passed** their last inspection, where a status rule has no signal.
* Scores are calibrated probabilities with per-building explanations (TreeSHAP).
* IBM Z NNPA acceleration is enabled only after a numerical parity check against the CPU model. The section above reports the actual kernel and engine; local execution is not evidence of LinuxONE execution.

*Decision support only: a field inspection is still required. The score ranks buildings for review; it is not a structural simulation.*